# NBME — Stage 3 & 4: Student Model Training + Inference

**Stage 3**: Student model training with pseudo labels + real data, 3-channel auxiliary targets, AWP  
**Stage 4**: CV evaluation, post-processing, ensemble with Optuna weight search

### Prerequisites (verify before running)
1. `train.csv` — real labeled data (columns: `id, case_num, pn_num, feature_num, annotation, location`)
2. `patient_notes.csv` — columns: `pn_num, case_num, pn_history`
3. `features.csv` — columns: `feature_num, case_num, feature_text`
4. `final_ensemble_pred_missing_rows.csv` — teacher pseudo labels (your teammate's output)
5. Pretrained teacher/base model weights in `./models/` directory

## Configuration

In [1]:
import gc, re, ast, warnings
import numpy as np
import pandas as pd
from pathlib import Path
from itertools import chain
from tqdm.auto import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.cuda.amp import autocast, GradScaler
from sklearn.model_selection import train_test_split

import optuna

from transformers import (
    AutoTokenizer, AutoModel, AutoConfig,
    get_linear_schedule_with_warmup,
    get_cosine_schedule_with_warmup
)
from sklearn.model_selection import GroupKFold, StratifiedKFold

pd.set_option('display.max_colwidth', 100)

print(torch.__version__)
print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0))

2.12.0.dev20260407+cu128
True
NVIDIA GeForce RTX 5090


### CFG configuration

In [2]:
class CFG:
    DATA_DIR = Path('.')
    PSEUDO_LABEL_PATH = Path('./final_ensemble_pred_missing_rows.csv')
    OUTPUT_DIR = Path('./outputs')
    
    model_name = './deberta-base'
    max_len = 512
    # As defined, three channels: Normal, Start, End
    num_labels = 3                   
     # only train 4/10 folds
    n_folds = 10
    train_folds = [0]
    epochs = 1
    # effective batch = batch_size * grad_accum
    batch_size = 8 
    grad_accum = 4                         
    lr = 2e-5
    weight_decay = 1e-2
    warmup_ratio = 0.1
    max_grad_norm = 1.0
    scheduler = 'cosine'
    fp16 = True
    seed = 42

    # AWP
    use_awp = False
    awp_lr = 1.0
    awp_eps = 0.01
    awp_start_epoch = 2                   

    # filter pseudo labels below the threshold
    pseudo_confidence_threshold = 0.80
    
    # 10% real labels, 90% pseudo labels in each batch
    real_data_ratio = 0.10                      

    #  prob of dropping a sentence of real data
    aug_sentence_drop_p = 0.2                  

    # joint-probability threshold
    threshold = 0.3      
    # merge adjacent spans within N chars                
    merge_gap = 3                         

    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    num_workers = 0

CFG.OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print(f'Device: {CFG.device}')
print(f'Model:  {CFG.model_name}')

Device: cuda
Model:  ./deberta-base


### Code reproductivity
fixes all random seeds to ensure fully reproducible results.

In [3]:
def seed(seed):
    import random
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

seed(CFG.seed)

## Data pre-processing

### Load real data

In [4]:

train_df = pd.read_csv(CFG.DATA_DIR / 'train.csv')
pn_df = pd.read_csv(CFG.DATA_DIR / 'patient_notes.csv')
feature_df = pd.read_csv(CFG.DATA_DIR / 'features.csv')

# Merge patient note text and feature text into train_df
train_df = train_df.merge(pn_df[['pn_num', 'case_num', 'pn_history']], on=['pn_num', 'case_num'], how='left')
train_df = train_df.merge(feature_df[['feature_num', 'case_num', 'feature_text']], on=['feature_num', 'case_num'], how='left')

# Transfrom data
# Parse location strings like "'0 3; 5 9'" -> list of (start, end) tuples
def parse_location(loc_str):
    """Convert location string to list of (start, end) int tuples."""
    # Return empty list if the location is missing or empty
    if pd.isna(loc_str) or loc_str in ('', '[]'):
        return []
    
    # Handle two formats:
    # Format 1: list-of-strings ["0 3", "5 9"] → use ast.literal_eval to parse
    # Format 2: raw string "0 3;5 9" → split by semicolon
    if loc_str.startswith('['):
        spans = ast.literal_eval(loc_str)
    else:
        spans = [s.strip() for s in loc_str.split(';')]
    result = []
    for span in spans:
        # Remove whitespace and quotes
        span = span.strip().strip("'")
        if span:
            # Split "0 3" into ["0", "3"] and convert to integers
            parts = span.split()
            if len(parts) == 2:
                result.append((int(parts[0]), int(parts[1])))
    # Example return element: [(0, 3), (5, 9)]
    return result

train_df['location_parsed'] = train_df['location'].apply(parse_location)
train_df['is_real'] = True

print(f'The number of train samples: {len(train_df)}')
train_df[['id', 'pn_num', 'feature_num', 'pn_history', 'feature_text', 'location_parsed', 'is_real']].head(3)

The number of train samples: 14300


,id,pn_num,feature_num,pn_history,feature_text,location_parsed,is_real
0,00016_000,16,0,"HPI: 17yo M presents with palpitations. Patient reports 3-4 months of intermittent episodes of ""...",Family-history-of-MI-OR-Family-history-of-myocardial-infarction,"[(696, 724)]",True
1,00016_001,16,1,"HPI: 17yo M presents with palpitations. Patient reports 3-4 months of intermittent episodes of ""...",Family-history-of-thyroid-disorder,"[(668, 693)]",True
2,00016_002,16,2,"HPI: 17yo M presents with palpitations. Patient reports 3-4 months of intermittent episodes of ""...",Chest-pressure,"[(203, 217)]",True


### Pseudo Labels preprocessing

In [5]:
pseudo_df = pd.read_csv(CFG.PSEUDO_LABEL_PATH)
print(f'Pseudo label samples: {len(pseudo_df)}')
print(f'Confidence distribution:\n{pseudo_df["confidence"].describe().round(3)}')

Pseudo label samples: 612602
Confidence distribution:
count    612602.000
mean          0.682
std           0.420
min           0.000
25%           0.085
50%           0.957
75%           0.985
max           0.996
Name: confidence, dtype: float64


### Filter & Prepare Pseudo Labels
1. Keep only high-confidence pseudo labels (≥ 0.80)
2. merge patient note and feature text features so the format matches the real data.

In [6]:
# Keep rows with non-empty annotations AND confidence >= threshold
pseudo_filtered = pseudo_df[
    (pseudo_df['annotation'] != '[]') &
    (pseudo_df['confidence'] >= CFG.pseudo_confidence_threshold)
].copy()

print(f'After filtering (conf >= {CFG.pseudo_confidence_threshold}): {len(pseudo_filtered)} rows')

# Merge patient note and feature text
pseudo_filtered = pseudo_filtered.merge(
    pn_df[['pn_num', 'case_num', 'pn_history']], on=['pn_num', 'case_num'], how='left'
)
pseudo_filtered = pseudo_filtered.merge(
    feature_df[['feature_num', 'case_num', 'feature_text']], on=['feature_num', 'case_num'], how='left'
)

# Parse location string → list of (start, end) tuples
pseudo_filtered['location_parsed'] = pseudo_filtered['location'].apply(parse_location)

# Mark as pseudo (not real) so augmentation is skipped later
pseudo_filtered['is_real'] = False

# use pseudo id as string to match train_df for id column
pseudo_filtered = pseudo_filtered.rename(columns={'id': 'id'})

print(f'Pseudo label samples after merge: {len(pseudo_filtered)}')
pseudo_filtered[['id', 'pn_num', 'feature_num', 'confidence', 'location_parsed']].head(3)

After filtering (conf >= 0.8): 405594 rows
Pseudo label samples after merge: 405594


,id,pn_num,feature_num,confidence,location_parsed
0,pred_0,0,0,0.969671,"[(553, 566)]"
1,pred_1,0,1,0.957691,"[(576, 597)]"
2,pred_3,0,3,0.925508,"[(216, 228)]"


### Combine Real + Pseudo Data
Merge real data with filtered pseudo labels (90% pseudo, 10% real).

In [7]:
# Align columns before concat
SHARED_COLS = ['id', 'case_num', 'pn_num', 'feature_num', 
               'pn_history', 'feature_text', 'location_parsed', 'is_real', 'confidence']

# real labels have confidence = 1.0
train_df['confidence'] = 1.0   

combined_df = pd.concat(
    [train_df[SHARED_COLS], pseudo_filtered[SHARED_COLS]], 
    ignore_index=True
)

print(f'Combined dataset (real + pseudo): {len(combined_df)} rows')
print(f'Real: {combined_df["is_real"].sum()}')
print(f'Pseudo: {(~combined_df["is_real"]).sum()}')

Combined dataset (real + pseudo): 419894 rows
Real: 14300
Pseudo: 405594


## Model Components

Core blocks of the student model pipeline: tokenisation, label construction, the PyTorch Dataset wrapper, the DeBERTa-based model architecture, the 3-channel BCE loss function, and the Adversarial Weight Perturbation (AWP) training technique.

### Tokenization & 3-Channel Label Construction
Tokenize each (feature, patient_note) pair and build token-level labels across 3 channels.
For pseudo labels we use soft labels rather than hard 0/1.

We use a label vector `[normal, start, end]` for each token position:
- **Channel 0 (Normal)**: 1.0 if token is inside any annotated span
- **Channel 1 (Start)**: 1.0 if token is the *first* token of a span  
- **Channel 2 (End)**: 1.0 if token is the *last* token of a span

In [8]:
tokenizer = AutoTokenizer.from_pretrained(CFG.model_name)


# Augmentation on real data
def drop_random_sentence(text, location):
    """
    Randomly drop one sentence from text that does NOT overlap with any annotated span.
    If no safe sentence found, return text unchanged. This is to prevent overfitting to the training data.
    """
    # split text into sentences by punctuation followed by whitespace
    sentences = re.split(r'(?<=[.!?])\s+', text)
    
    # If only one sentence, nothing to drop
    if len(sentences) <= 1:
        return text

    # a set of all character indexes that are inside an annotated span
    annotated = set()
    for (s, e) in location:
        annotated.update(range(s, e))

    # find out sentences that don't overlap with any annotated span
    pos = 0
    candidates = []
    for sent in sentences:
        s_start = pos
        s_end   = pos + len(sent)
        sent_chars = set(range(s_start, s_end))
        # Only add if this sentence has no annotated characters
        if not sent_chars.intersection(annotated):
            candidates.append(sent)
        # +1 for the space consumed by split
        pos = s_end + 1  

    if not candidates:
        return text

    # randomly pick one candidate sentence and remove it from text
    drop = np.random.choice(candidates)
    return re.sub(re.escape(drop) + r'\s*', '', text, count=1).strip()


def tokenize_and_build_labels(row, tokenizer, max_len=CFG.max_len):
    """
    Tokenize [feature_text] + [pn_history] and build 3-channel per-token labels.
    Returns dict with input_ids, attention_mask, token_type_ids, labels, offset_mapping.
    """
    pn_text = str(row['pn_history'])
    feature_text = str(row['feature_text'])
    
    # list of (start, end) char tuples
    location = row['location_parsed']   
    is_real = row['is_real']
    confidence = float(row['confidence'])

    # Augmentation on real data
    if is_real and np.random.random() < CFG.aug_sentence_drop_p:
        pn_text = drop_random_sentence(pn_text, location)

    # Tokenization
    # "fever" + "patient has fever and cough"
    # [CLS] fever [SEP] patient has fever and cough [SEP]
    encoding = tokenizer(
        feature_text,
        pn_text,
        max_length=max_len,
        padding='max_length',
        truncation=True,
        return_offsets_mapping=True,
        return_token_type_ids=True,
    )

    # list of (char_start, char_end) per token
    offset_mapping = encoding['offset_mapping']   
    
    # 0=feature tokens, 1=pn tokens, None=special 
    sequence_ids = encoding.sequence_ids()       

    # 3-channel float labels — shape (max_len, 3)
    labels = np.zeros((max_len, 3), dtype=np.float32)

    # Character level boolean arrays for the patient note text
    # 1 if it is in the span, 0 otherwise
    pn_len = len(pn_text)
    char_normal = np.zeros(pn_len, dtype=np.float32)
    char_start = np.zeros(pn_len, dtype=np.float32)
    char_end = np.zeros(pn_len, dtype=np.float32)
    
    # Soft labels for pseudo labels, hard 1.0 for real labels
    label_value = confidence if not is_real else 1.0

    for (s, e) in location:
        s, e = int(s), int(e)
        if s >= pn_len or e > pn_len:
            continue
        char_normal[s:e]  = label_value
        if s < pn_len:
            # first char of span
            char_start[s] = label_value    
        if e - 1 >= 0 and e - 1 < pn_len:
            # last char of span
            char_end[e - 1] = label_value  

    # Map char-level labels to token-level labels
    # Only process pn tokens (seq_id=1); skip feature tokens and special tokens
    for idx, (seq_id, (tok_s, tok_e)) in enumerate(zip(sequence_ids, offset_mapping)):
        if seq_id != 1:
            continue
        if tok_s == tok_e:
            continue
        # If any character in this token's range is labelled, mark the token
        labels[idx, 0] = char_normal[tok_s:tok_e].max()   
        labels[idx, 1] = char_start[tok_s:tok_e].max()    
        labels[idx, 2] = char_end[tok_s:tok_e].max()     

    return {
        'input_ids': np.array(encoding['input_ids'], dtype=np.int64),
        'attention_mask': np.array(encoding['attention_mask'], dtype=np.int64),
        'token_type_ids': np.array(encoding['token_type_ids'], dtype=np.int64),
        'labels': labels,          # (max_len, 3)
        'offset_mapping': np.array(offset_mapping, dtype=np.int32),
        'sequence_ids': np.array([s if s is not None else -1 for s in sequence_ids], dtype=np.int8),
    }


### Dataset
Wraps the dataframe rows into a format the DataLoader can iterate over.

In [9]:
class NBMEDataset(Dataset):
    """
    Handles both real and pseudo-labeled rows.
    For training: returns labels (3-channel).
    For inference (labels=False): skips label computation.
    """
    def __init__(self, df, tokenizer, is_train=True):
        self.df = df.reset_index(drop=True)
        self.tokenizer = tokenizer
        self.is_train = is_train

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        encoded = tokenize_and_build_labels(row, self.tokenizer)

        item = {
            'input_ids': torch.tensor(encoded['input_ids'], dtype=torch.long),
            'attention_mask': torch.tensor(encoded['attention_mask'], dtype=torch.long),
            'token_type_ids': torch.tensor(encoded['token_type_ids'], dtype=torch.long),
            'offset_mapping': torch.tensor(encoded['offset_mapping'], dtype=torch.long),
            'sequence_ids': torch.tensor(encoded['sequence_ids'],   dtype=torch.long),
        }
        if self.is_train:
            item['labels'] = torch.tensor(encoded['labels'], dtype=torch.float32)
        return item

### 3 channels token classifier
Backbone model with a 3-output linear head. Each token produces three probabilities: Normal, Start, End.

In [10]:
class NBMEModel(nn.Module):
    """
    DeBERTa-based token classifier with 3 output channels:
    Channel 0: Normal (is this token part of the clinical feature?)
    Channel 1: Start (is this token the START of the feature span?)
    Channel 2: End (is this token the END of the feature span?)
    """
    def __init__(self, model_name, num_labels=3):
        super().__init__()
        config = AutoConfig.from_pretrained(model_name)
        config.hidden_dropout_prob       = 0.0
        config.attention_probs_dropout_prob = 0.0

        self.backbone = AutoModel.from_pretrained(model_name, config=config)
        self.dropout  = nn.Dropout(0.1)
        self.head = nn.Linear(config.hidden_size, num_labels)

    def forward(self, input_ids, attention_mask, token_type_ids):
        out = self.backbone(
            input_ids=input_ids,
            attention_mask=attention_mask,
            token_type_ids=token_type_ids,
        )
        # (B, L, hidden)
        seq_out = out.last_hidden_state          
        seq_out = self.dropout(seq_out)
        # (B, L, 3)
        logits  = self.head(seq_out)             
        return logits                         

### Loss Function
Binary cross-entropy applied. Padding and special tokens are masked out. Supports soft labels for pseudo data.

In [11]:
class ThreeChannelBCELoss(nn.Module):
    """
    Binary cross-entropy loss over all 3 channels simultaneously.
    Handles soft labels (confidence < 1.0) naturally via BCEWithLogitsLoss.
    Masks out special tokens and padding (sequence_id != 1).
    """
    def __init__(self):
        super().__init__()
        self.bce = nn.BCEWithLogitsLoss(reduction='none')

    def forward(self, logits, labels, sequence_ids):
        """
        logits: (B, L, 3)
        labels: (B, L, 3), soft label [0,1] values
        sequence_ids: (B, L), 1 for pn tokens
        """
        # (B, L, 3)
        loss = self.bce(logits, labels)           
        # (B, L, 1) broadcasts over channels
        mask = (sequence_ids == 1).unsqueeze(-1)  
        loss = (loss * mask).sum() / mask.sum().clamp(min=1)
        return loss

criterion = ThreeChannelBCELoss()

### AWP (Adversarial Weight Perturbation)
Perturbs model weights in the direction of maximum loss to force the model to improve generalization performance.

In [12]:
class AWP:
    """
    Adversarial Weight Perturbation.
    1. Normal forward+backward → compute clean gradients
    2. Perturb weights by adding scaled gradient noise (makes the model "worst case")
    3. Forward+backward again on perturbed weights → get adversarial gradients
    4. Restore original weights; optimizer.step() uses adversarial gradients
    """
    def __init__(self, model, optimizer, adv_lr=1.0, adv_eps=0.01):
        self.model = model
        self.optimizer = optimizer
        # step size for computing perturbation
        self.adv_lr = adv_lr
        # max allowed perturbation magnitude
        self.adv_eps = adv_eps
        # stores original weights
        self.backup = {}          
        self.backup_eps= {}

    def attack(self):
        """Compute and apply weight perturbation"""
        # small epsilon to avoid division by zero
        e1 = 1e-6
        for name, param in self.model.named_parameters():
            if param.requires_grad and param.grad is not None:
                # restore original weight later
                self.backup[name] = param.data.clone()
                # Compute gradient norm 
                norm = torch.norm(param.grad)
                if norm != 0 and not torch.isnan(norm):
                    # aompute perturbation direction: scaled gradient unit vector
                    r_adv = self.adv_lr * param.grad / (norm + e1)
                    # apply perturbation to weights
                    param.data.add_(r_adv)
                    # clamp perturbation to stay within epsilon range
                    param.data = self._clamp(param.data, self.backup[name], self.adv_eps)

    def restore(self):
        """Restore original weights after adversarial step."""
        for name, param in self.model.named_parameters():
            if name in self.backup:
                # Reset weight back to pre-attack value
                param.data = self.backup[name]
        # clear backup
        self.backup = {}

    @staticmethod
    def _clamp(data, original, eps):
        # ensure perturbed weight stays within [original-eps, original+eps]
        return torch.clamp(data, original - eps, original + eps)
    

### Training and Validation functions

In [13]:
def train_one_epoch(model, loader, optimizer, scheduler, scaler, awp, epoch):
    model.train()
    total_loss = 0.0
    use_awp_this_epoch = CFG.use_awp and (epoch >= CFG.awp_start_epoch)

    optimizer.zero_grad()

    for step, batch in enumerate(tqdm(loader, desc=f'Epoch {epoch} train')):
        # token ids
        input_ids      = batch['input_ids'].to(CFG.device)
        # 1=real token, 0=padding
        attention_mask = batch['attention_mask'].to(CFG.device)
        # 0=feature, 1=patient note
        token_type_ids = batch['token_type_ids'].to(CFG.device)
        # (B, L, 3) ground truth
        labels         = batch['labels'].to(CFG.device)
        # used to mask loss
        sequence_ids   = batch['sequence_ids'].to(CFG.device)

        is_accum_step = ((step + 1) % CFG.grad_accum == 0)

        # normal forward pass
        with torch.amp.autocast('cuda', enabled=CFG.fp16):
            logits = model(input_ids, attention_mask, token_type_ids)
            loss   = criterion(logits, labels, sequence_ids)
            loss   = loss / CFG.grad_accum

        scaler.scale(loss).backward()

        # only update model every grad_accum steps
        if is_accum_step:
            if use_awp_this_epoch:
                # unscale once before AWP attack
                scaler.unscale_(optimizer)
                awp.attack()
                optimizer.zero_grad()
                # AWP adversarial forward in fp32, no scaler
                adv_logits = model(input_ids, attention_mask, token_type_ids)
                adv_loss   = criterion(adv_logits, labels, sequence_ids)
                adv_loss.backward()
                awp.restore()
                # clip and step without unscaling again
                nn.utils.clip_grad_norm_(model.parameters(), CFG.max_grad_norm)
                scaler.step(optimizer)
            else:
                scaler.unscale_(optimizer)
                nn.utils.clip_grad_norm_(model.parameters(), CFG.max_grad_norm)
                scaler.step(optimizer)

            scaler.update()
            optimizer.zero_grad()
            scheduler.step()

        # accumulate loss
        total_loss += loss.item() * CFG.grad_accum

    # average loss
    return total_loss / len(loader)


@torch.no_grad()
def validate(model, loader):
    """
    Return:
      all_preds: list of (B, L, 3) sigmoid arrays
      all_labels: list of (B, L, 3) ground truth arrays
      all_offset_mappings: list of (B, L, 2) arrays
      all_sequence_ids: list of (B, L) arrays
    """
    model.eval()
    all_preds, all_labels, all_offsets, all_seqids = [], [], [], []

    for batch in tqdm(loader, desc='Validation'):
        input_ids = batch['input_ids'].to(CFG.device)
        attention_mask = batch['attention_mask'].to(CFG.device)
        token_type_ids = batch['token_type_ids'].to(CFG.device)

        with torch.amp.autocast('cuda', enabled=CFG.fp16):
            logits = model(input_ids, attention_mask, token_type_ids)

        # (B, L, 3)
        preds = torch.sigmoid(logits).cpu().numpy()          
        all_preds.append(preds)
        # (B, L, 3)
        all_labels.append(batch['labels'].numpy())      
         # (B, L, 2)     
        all_offsets.append(batch['offset_mapping'].numpy()) 
         # (B, L)
        all_seqids.append(batch['sequence_ids'].numpy())    

    return (
        np.concatenate(all_preds,   axis=0),
        np.concatenate(all_labels,  axis=0),
        np.concatenate(all_offsets, axis=0),
        np.concatenate(all_seqids,  axis=0),
    )

## Post Processing: Convert probabilities to Character Spans

**Joint probability**: `score[t] = prob_normal[t] * (prob_start[t] + prob_end[t]) / 2`  

In [14]:
def preds_to_char_spans(probs_3ch, offset_mapping, sequence_ids,
                         threshold=CFG.threshold, merge_gap=CFG.merge_gap):
    """
    Convert per-token 3-channel probabilities to a list of character span strings.
    Args:
        probs_3ch: np.array (L, 3)  — sigmoid probabilities [normal, start, end]
        offset_mapping: np.array (L, 2) — (char_start, char_end) per token
        sequence_ids: np.array (L,) — 1 for pn tokens
        threshold: float — joint prob threshold
        merge_gap: int — merge spans within this many characters
    """
    # extract each channel from the 3-channel probability array
    prob_normal = probs_3ch[:, 0]
    prob_start = probs_3ch[:, 1]
    prob_end = probs_3ch[:, 2]

    # joint score: normal × mean(start, end)
    # If model only has 1 channel, just use normal
    boundary_score = (prob_start + prob_end) / 2.0
    joint_score = prob_normal * boundary_score

    # fall back to normal-only if boundary channels are all zero
    if boundary_score.max() < 1e-4:
        joint_score = prob_normal

    # keep tokens that exceed threshold AND belong to patient note 
    active = (joint_score >= threshold) & (sequence_ids == 1)

    char_spans = []
    for i, is_active in enumerate(active):
        if not is_active:
            continue
        tok_s, tok_e = int(offset_mapping[i, 0]), int(offset_mapping[i, 1])
        if tok_s == tok_e:
            continue
        char_spans.append((tok_s, tok_e))

    if not char_spans:
        return ''

    char_spans.sort()
    merged = [char_spans[0]]
    for s, e in char_spans[1:]:
        prev_s, prev_e = merged[-1]
        # if this span starts within merge_gap chars of the previous, merge them
        if s <= prev_e + merge_gap:
            merged[-1] = (prev_s, max(prev_e, e))
        else:
            merged.append((s, e))

    # return spans as semicolon-separated string e.g. "0 5;12 17"
    return ';'.join(f'{s} {e}' for s, e in merged)


def strip_whitespace_from_spans(span_str, pn_text):
    """
    Post-processing: trim leading/trailing whitespace within each span
    by adjusting character indices to exclude whitespace chars.
    
    This is the critical post-processing step from the 1st place solution.
    """
    if not span_str:
        return ''

    result = []
    for part in span_str.split(';'):
        part = part.strip()
        if not part:
            continue
        s, e = int(part.split()[0]), int(part.split()[1])
        # move start index to a non-whitespace char
        while s < e and pn_text[s].isspace():
            s += 1
        # move end index to a non-whitespace char
        while e > s and pn_text[e - 1].isspace():
            e -= 1
        # keep span if it still has content after trimming
        if s < e:
            result.append(f'{s} {e}')

    # rejoin all cleaned spans with semicolons
    return ';'.join(result)

## F1 Score Evaluation

In [15]:
def parse_span_str(span_str):
    """Parse '0 3;5 9' → set of integer char indices {0,1,2,5,6,7,8}"""
    indices = set()
    if not span_str or pd.isna(span_str):
        return indices
    for part in str(span_str).split(';'):
        part = part.strip()
        if part:
            s, e = int(part.split()[0]), int(part.split()[1])
            indices.update(range(s, e))
    return indices


def compute_micro_f1(pred_spans, true_spans):
    """
    Compute micro-averaged F1 across all samples.
    Args:
        pred_spans: list of span strings (predictions)
        true_spans: list of span strings (ground truth)
    """
    total_tp = total_fp = total_fn = 0

    for pred, true in zip(pred_spans, true_spans):
        pred_set = parse_span_str(pred)
        true_set = parse_span_str(true)

        tp = len(pred_set & true_set)
        fp = len(pred_set - true_set)
        fn = len(true_set - pred_set)

        total_tp += tp
        total_fp += fp
        total_fn += fn

    precision = total_tp / (total_tp + total_fp + 1e-9)
    recall = total_tp / (total_tp + total_fn + 1e-9)
    f1 = 2 * precision * recall / (precision + recall + 1e-9)

    return {'tp': total_tp, 'fp': total_fp, 'fn': total_fn,
            'precision': precision, 'recall': recall, 'f1': f1}

## CV Split (GroupStratifiedKFold)

The 1st place solution uses **10-fold GroupStratifiedKFolds** — grouping by `pn_num` (same patient note never split across folds) and stratifying by `case_num` (balanced case distribution per fold).

In [16]:
def create_cv_folds(df, n_folds=CFG.n_folds):
    """
    GroupKFold on pn_num ensures no pn_history leaks between train and val.
    We also stratify by case_num to keep case distribution balanced.
    """
    df = df.copy()
    # initialize all rows as unassigned
    df['fold'] = -1
    # ensures all rows with the same pn_num go to the same fold
    # prevents the same patient note appearing in both train and val
    gkf = GroupKFold(n_splits=n_folds)
    for fold_idx, (_, val_idx) in enumerate(
        gkf.split(df, groups=df['pn_num'])
    ):
        # assign fold number to validation rows for this split
        df.loc[df.index[val_idx], 'fold'] = fold_idx
    # real samples distribution in each fold
    print('Fold distribution:')
    print(df.groupby('fold')['is_real'].agg(['sum', 'count']).rename(
        columns={'sum': 'real', 'count': 'total'}
    ))
    return df

# only assign folds to real data, pseudo labels are used for training only
real_df = combined_df[combined_df['is_real']].copy()

# all unique patient note IDs
unique_pn = real_df['pn_num'].unique()
# split patient notes into 80% train and 20% test
train_pn, test_pn = train_test_split(unique_pn, test_size=0.2, random_state=CFG.seed)
# select rows belonging to train/test patient notes
coursework_train = real_df[real_df['pn_num'].isin(train_pn)].copy()
coursework_test  = real_df[real_df['pn_num'].isin(test_pn)].copy()
print(f'Train: {len(coursework_train)}')
print(f'Test:  {len(coursework_test)}')

real_df = create_cv_folds(coursework_train)

# put pseudo labels into its dataframe
pseudo_only_df = combined_df[~combined_df['is_real']].copy()
print(f'\nReal: {len(real_df)} | Pseudo: {len(pseudo_only_df)}')

Train: 11342
Test:  2958
Fold distribution:
      real  total
fold             
0     1134   1134
1     1134   1134
2     1135   1135
3     1135   1135
4     1135   1135
5     1135   1135
6     1134   1134
7     1134   1134
8     1133   1133
9     1133   1133

Real: 11342 | Pseudo: 405594


### Main Training Loop

In [17]:
def get_optimizer_and_scheduler(model, num_training_steps):
    """AdamW with differential learning rates."""
    no_decay = ['bias', 'LayerNorm.weight', 'LayerNorm.bias']

    param_groups = [
        # backbone: lower lr, weight decay
        {'params': [p for n, p in model.backbone.named_parameters()
                    if not any(nd in n for nd in no_decay)],
         'lr': CFG.lr, 'weight_decay': CFG.weight_decay},
        # backbone parameters without weight decay
        {'params': [p for n, p in model.backbone.named_parameters()
                    if any(nd in n for nd in no_decay)],
         'lr': CFG.lr, 'weight_decay': 0.0},
        # classification head uses 10x higher lr to learn faster than backbone
        {'params': model.head.parameters(),
         'lr': CFG.lr * 10, 'weight_decay': 0.0},
    ]

    optimizer = torch.optim.AdamW(param_groups)

    # warmup: gradually increase lr from 0 to CFG.lr
    num_warmup = int(num_training_steps * CFG.warmup_ratio)
    # lr decreases linearly after warmup
    if CFG.scheduler == 'cosine':
        scheduler = get_cosine_schedule_with_warmup(
            optimizer, num_warmup_steps=num_warmup,
            num_training_steps=num_training_steps
        )
    else:
        scheduler = get_linear_schedule_with_warmup(
            optimizer, num_warmup_steps=num_warmup,
            num_training_steps=num_training_steps
        )
    return optimizer, scheduler


def train_fold(fold, real_df, pseudo_df):
    print(f' FOLD {fold}')

    # split real data into current fold is validation and rest is training
    val_real = real_df[real_df['fold'] == fold]
    train_real = real_df[real_df['fold'] != fold]

    # combine real training data with pseudo labels and shuffle
    train_combined = pd.concat([train_real, pseudo_df], ignore_index=True)
    train_combined = train_combined.sample(frac=1, random_state=CFG.seed).reset_index(drop=True)

    print(f'Train: {len(train_combined)} (real: {len(train_real)}, pseudo: {len(pseudo_df)})')
    print(f'Val: {len(val_real)} (real only)')
    
    train_ds = NBMEDataset(train_combined, tokenizer, is_train=True)
    val_ds = NBMEDataset(val_real, tokenizer, is_train=True)

    train_loader = DataLoader(
        train_ds, batch_size=CFG.batch_size, shuffle=True,
        num_workers=CFG.num_workers, pin_memory=True
    )
    
    # use 2x batch size for validation since no gradients needed
    val_loader = DataLoader(
        val_ds, batch_size=CFG.batch_size * 2, shuffle=False,
        num_workers=CFG.num_workers, pin_memory=True
    )

    # initialize model, optimizer, scheduler, scaler and AWP
    model = NBMEModel(CFG.model_name, num_labels=CFG.num_labels).to(CFG.device)
    num_steps = len(train_loader) * CFG.epochs // CFG.grad_accum
    optimizer, scheduler = get_optimizer_and_scheduler(model, num_steps)
    scaler = torch.amp.GradScaler('cuda', enabled=CFG.fp16)
    awp    = AWP(model, optimizer, adv_lr=CFG.awp_lr, adv_eps=CFG.awp_eps)

    best_f1    = 0.0
    best_path  = CFG.OUTPUT_DIR / f'model_fold{fold}.pt'

    for epoch in range(CFG.epochs):
        # get average loss
        train_loss = train_one_epoch(model, train_loader, optimizer, scheduler, scaler, awp, epoch)

        # run validation and collect probabilities
        preds, labels, offsets, seq_ids = validate(model, val_loader)

        # convert probabilities to span strings and compare with ground truth
        pred_spans = []
        true_spans = []
        for i, row in enumerate(val_real.itertuples()):
            span_str = preds_to_char_spans(preds[i], offsets[i], seq_ids[i])
            span_str = strip_whitespace_from_spans(span_str, str(row.pn_history))
            pred_spans.append(span_str)
            true_spans.append(';'.join(f'{s} {e}' for s, e in row.location_parsed))

        # compute micro F1
        metrics = compute_micro_f1(pred_spans, true_spans)
        f1 = metrics['f1']

        # tag epoch log with AWP indicator if active
        awp_tag = '(+AWP)' if CFG.use_awp and epoch >= CFG.awp_start_epoch else ''
        print(f'Epoch {epoch}: loss={train_loss:.4f}  F1={f1:.4f}  '
              f'P={metrics["precision"]:.4f}  R={metrics["recall"]:.4f}  {awp_tag}')

        # Save checkpoint for the best F1
        if f1 > best_f1:
            best_f1 = f1
            torch.save(model.state_dict(), best_path)
            print(f'  ✓ Saved best model (F1={best_f1:.4f})')

    # free GPU memory before next fold
    del model, optimizer, scheduler, scaler, awp, train_loader, val_loader
    torch.cuda.empty_cache()
    gc.collect()

    print(f'Fold {fold} best CV F1: {best_f1:.4f}')
    return best_f1

### Run training across selected folds

In [ ]:
# train one model per fold and collect validation F1 scores
fold_scores = {}

print(f"train fold is: {CFG.train_folds}")
for fold in CFG.train_folds:
    # train all folds except current, validate on current fold
    score = train_fold(fold, real_df, pseudo_only_df)
    # store best F1 score
    fold_scores[fold] = score

print(f'fold scores: {fold_scores}')
print('CROSS-VALIDATION RESULTS')
for fold, score in fold_scores.items():
    print(f'Fold {fold}: F1 = {score:.4f}')
    
# key metric: Mean CV F1
mean_cv = np.mean(list(fold_scores.values()))
print(f'\n  Mean CV F1: {mean_cv:.4f}')

train fold is: [0]
 FOLD 0
Train: 415802 (real: 10208, pseudo: 405594)
Val: 1134 (real only)


Epoch 0 train:   0%|          | 0/51976 [00:00<?, ?it/s]

/tmp/ipykernel_39979/1376869830.py:52: UserWarning: Detected call of `lr_scheduler.step()` before `optimizer.step()`. In PyTorch 1.1.0 and later, you should call them in the opposite order: `optimizer.step()` before `lr_scheduler.step()`.  Failure to do this will result in PyTorch skipping the first value of the learning rate schedule. See more details at https://pytorch.org/docs/stable/optim.html#how-to-adjust-learning-rate
  scheduler.step()


## Inference on Test Set

In [ ]:
# disable gradient computation to save memory during inference
@torch.no_grad()
def run_inference(model_path, test_df):
    """
    Run inference with a saved model checkpoint.
    Returns np.array of shape (N, max_len, 3) probabilities.
    """
    
    # load model and restore saved weights
    model = NBMEModel(CFG.model_name, num_labels=CFG.num_labels)
    model.load_state_dict(torch.load(model_path, map_location=CFG.device))
    model.to(CFG.device)
    model.eval()

    # Add dummy columns so tokenize_and_build_labels doesn't crash
    # (inference data has no labels, so we fill with empty/default values)
    inf_df = test_df.copy()
    if 'location_parsed' not in inf_df.columns:
        inf_df['location_parsed'] = [[] for _ in range(len(inf_df))]
    if 'is_real' not in inf_df.columns:
        # treat as pseudo to skip augmentation
        inf_df['is_real'] = False
    if 'confidence' not in inf_df.columns:
        inf_df['confidence'] = 0.0

    # dataset and dataloader
    ds = NBMEDataset(inf_df, tokenizer, is_train=False)
    loader = DataLoader(ds, batch_size=CFG.batch_size * 2, shuffle=False,
                        num_workers=CFG.num_workers, pin_memory=True)

    all_probs = []
    all_offsets = []
    all_seq_ids = []

    for batch in tqdm(loader, desc=f'Inference: {Path(model_path).name}'):
        input_ids      = batch['input_ids'].to(CFG.device)
        attention_mask = batch['attention_mask'].to(CFG.device)
        token_type_ids = batch['token_type_ids'].to(CFG.device)

        with torch.amp.autocast('cuda', enabled=CFG.fp16):
            logits = model(input_ids, attention_mask, token_type_ids)

        # apply sigmoid to convert logits to probabilities within the range [0, 1]
        probs = torch.sigmoid(logits).cpu().numpy()
        all_probs.append(probs)
        all_offsets.append(batch['offset_mapping'].numpy())
        all_seq_ids.append(batch['sequence_ids'].numpy())

    # free GPU memory
    del model
    torch.cuda.empty_cache()

    # merge all batches into single arrays of shape (N, max_len, 3)
    return (
        np.concatenate(all_probs,   axis=0),
        np.concatenate(all_offsets, axis=0),
        np.concatenate(all_seq_ids, axis=0),
    )

### Load test data

In [ ]:
test_df = pd.read_csv(CFG.DATA_DIR / 'test.csv')
test_df = test_df.merge(pn_df[['pn_num', 'case_num', 'pn_history']],  on=['pn_num', 'case_num'], how='left')
test_df = test_df.merge(feature_df[['feature_num', 'case_num', 'feature_text']], on=['feature_num', 'case_num'], how='left')

print(f'Test samples: {len(test_df)}')
test_df.head(3)

### Run inference for all trained folds

In [ ]:
# Collect test set predictions from each trained fold model
fold_probs_list = []

for fold in CFG.train_folds:
    model_path = CFG.OUTPUT_DIR / f'model_fold{fold}.pt'
    if not model_path.exists():
        print(f'Skipping fold {fold} — no checkpoint found')
        continue

    # inference on test set with this fold's model
    probs, offsets, seq_ids = run_inference(model_path, test_df)
    # accumulate probabilities, which will be averaged later for ensemble
    fold_probs_list.append(probs)

# Save offsets and seq_ids once for all folds
test_offsets = offsets
test_seq_ids = seq_ids

print(f'\nCollected predictions from {len(fold_probs_list)} fold models.')

## Ensemble with Optuna

Optuna searches for the best per-fold weights that maximize CV F1 on held-out real data.

In [ ]:
# Initialize arrays to store OOF predictions for all real data rows
oof_preds = np.zeros((len(real_df), CFG.max_len, 3), dtype=np.float32)
oof_offsets = np.zeros((len(real_df), CFG.max_len, 2), dtype=np.int32)
oof_seq_ids = np.zeros((len(real_df), CFG.max_len),    dtype=np.int8)

for fold in CFG.train_folds:
    model_path = CFG.OUTPUT_DIR / f'model_fold{fold}.pt'
    # Skip if this fold's model was not trained
    if not model_path.exists():
        continue

    # get the validation rows for this fold
    val_real_fold = real_df[real_df['fold'] == fold]
    # inference on this fold's validation data
    probs, offsets_f, seq_ids_f = run_inference(model_path, val_real_fold)

    # map fold indexes to their positions in the full real_df
    val_indices = real_df.index[real_df['fold'] == fold]
    pos = real_df.index.get_indexer(val_indices)
    
    # store predictions in the right places
    oof_preds[pos]   = probs
    oof_offsets[pos] = offsets_f
    oof_seq_ids[pos] = seq_ids_f

print('OOF probs max:', oof_preds.max())
print('OOF probs mean:', oof_preds.mean())

### Ensemble Search helper functions
Use Optuna to automatically search for the optimal decision threshold and per-model weights that maximise OOF F1.

In [ ]:
# Collect OOF and test predictions from all trained models
# shape: list of (N, L, 3)
all_model_oof_probs = [oof_preds]   
all_model_test_probs = [np.mean(fold_probs_list, axis=0)] 


# weighted average of multiple models' probability arrays
def ensemble_with_weights(probs_list, weights):
    weights = np.array(weights)
    # normalize weights to sum to 1
    weights = weights / weights.sum()
    combined = np.zeros_like(probs_list[0])
    for p, w in zip(probs_list, weights):
        # accumulate weighted probabilities
        combined += p * w
    return combined


# compute micro F1 on OOF data at a given threshold
def oof_f1_score(probs, threshold):
    pred_spans, true_spans = [], []
    for i, row in enumerate(real_df.itertuples()):
        # convert probabilities to span string with current threshold
        span_str = preds_to_char_spans(probs[i], oof_offsets[i], oof_seq_ids[i], threshold=threshold)
        span_str = strip_whitespace_from_spans(span_str, str(row.pn_history))
        pred_spans.append(span_str)
        # Convert ground truth tuples to span string for comparison
        true_spans.append(';'.join(f'{s} {e}' for s, e in row.location_parsed))
    return compute_micro_f1(pred_spans, true_spans)['f1']



def objective(trial):
    # Optuna suggests a weight per model and a threshold to try
    n_models = len(all_model_oof_probs)
    weights  = [trial.suggest_float(f'w{i}', 0.0, 1.0) for i in range(n_models)]
    threshold = trial.suggest_float('threshold', 0.3, 0.7)

    # model predictions with suggested weights and evaluate F1
    ensemble_probs = ensemble_with_weights(all_model_oof_probs, weights)
    f1 = oof_f1_score(ensemble_probs, threshold)
    return f1

# 200 trials to find the best weights and threshold
study = optuna.create_study(direction='maximize')
study.optimize(objective, n_trials=200, show_progress_bar=True)

# Extract best parameters found
best_params    = study.best_params
best_threshold = best_params['threshold']
best_weights   = [best_params[f'w{i}'] for i in range(len(all_model_oof_probs))]

print(f'\nOptuna best OOF F1: {study.best_value:.4f}')
print(f'Best threshold:     {best_threshold:.3f}')
print(f'Best weights:       {best_weights}')

<!-- ## 15. Build Submission -->

In [ ]:
# run inference on held-out test set with the saved best model
probs, offsets, seq_ids = run_inference(
    CFG.OUTPUT_DIR / 'model_fold0.pt', coursework_test
)

pred_spans, true_spans = [], []
for i, row in enumerate(coursework_test.itertuples()):
    # convert per-token probabilities to character span string e.g. "12 17;23 30"
    span_str = preds_to_char_spans(probs[i], offsets[i], seq_ids[i], threshold=best_threshold)
    # strm leading/trailing whitespace from each span boundary
    span_str = strip_whitespace_from_spans(span_str, str(row.pn_history))
    pred_spans.append(span_str)
    # convert ground truth tuples to same format for comparison
    true_spans.append(';'.join(f'{s} {e}' for s, e in row.location_parsed))

# Compare predictions vs ground truth at character level
test_metrics = compute_micro_f1(pred_spans, true_spans)
print(f'Test F1:      {test_metrics["f1"]:.4f}')
print(f'Test P:       {test_metrics["precision"]:.4f}')
print(f'Test Recall:  {test_metrics["recall"]:.4f}')

---
## Summary of Architecture

| Component | Implementation |
|-----------|----------------|
| **Data mixing** | 90% pseudo (conf ≥ 0.80) + 10% real per training run |
| **Soft labels** | Pseudo rows use `confidence` value instead of hard 1/0 |
| **3-channel labels** | `[normal, start, end]` per token; BCEWithLogitsLoss |
| **Augmentation** | p=0.2 random sentence drop on real data only |
| **AWP** | Starts at epoch 2; adv_lr=1.0, adv_eps=0.01 |
| **CV** | 10-fold GroupKFold on pn_num; train 5 folds |
| **Joint probability** | `prob_normal × mean(prob_start, prob_end)` |
| **Post-processing** | Strip leading/trailing whitespace from each span |
| **Ensemble** | Optuna weight search over OOF F1 (200 trials) |
| **Evaluation** | Micro-averaged F1 (char-level TP/FP/FN) |